In [1]:
# %% [markdown]
# # Includes

# %%
import pandas as pd
import numpy as np
import mne
import mne_nirs
import plotly.express as px

from pathlib import Path

from mne.preprocessing.nirs import (
    optical_density,
    beer_lambert_law,
    scalp_coupling_index,
    source_detector_distances,
    short_channels,
)
from mne.preprocessing.nirs import temporal_derivative_distribution_repair as tddr

from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.statistics import run_glm

/home/asunkari/miniconda3/envs/neuro-ml/lib/python3.12/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# %% [markdown]
# # Settings

# %%
root = Path.home() / "fnirs-representation-learning"
rs_data_dir = root / "snirf_dataset_2"
output_dir = root / "outputs"
output_tables_dir = output_dir / "tables"
output_figures_dir = output_dir / "figures"

output_tables_dir.mkdir(parents=True, exist_ok=True)
output_figures_dir.mkdir(parents=True, exist_ok=True)

subject_name = "Subj100"
subject_dir = rs_data_dir / subject_name

file_label = "hrf_50"
file_name = "resting_hrf_50.snirf"
file_path = subject_dir / file_name

short_separation_threshold_m = 0.015
long_separation_threshold_m = 0.025
sci_threshold = 0.50

filter_low_hz = 0.01
filter_high_hz = 0.20

epoch_tmin = -5.0
epoch_tmax = 30.0
baseline_window = (-5.0, 0.0)
response_window = (4.0, 8.0)

task_duration_s = 1.0
drift_high_pass_hz = 0.01

top_m_pairs = 5
fir_delays_s = list(range(26))

# keep this consistent with your existing notebooks for now
ppf_value = 0.1

pipeline_labels = ["Pref", "PnoSS", "PnoMC", "PFIR"]

## Helper functions

In [3]:
# %%
def get_cw_channel_indices(raw_snirf):
    picks_fnirs = mne.pick_types(raw_snirf.info, fnirs=True)
    channel_types = np.array(raw_snirf.get_channel_types())
    picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]
    return picks_cw

In [4]:
# %%
def build_cw_channel_table(raw_snirf, subject_name, file_label):
    picks_cw = get_cw_channel_indices(raw_snirf)
    cw_names = np.array(raw_snirf.ch_names)[picks_cw]

    distances_m = source_detector_distances(raw_snirf.info, picks=picks_cw)
    short_mask_all = short_channels(raw_snirf.info, threshold=short_separation_threshold_m)
    short_mask = short_mask_all[picks_cw]
    long_mask = distances_m >= long_separation_threshold_m

    pair_names = np.array([channel_name.split(" ")[0] for channel_name in cw_names])

    channel_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": cw_names,
        "pair_name": pair_names,
        "distance_m": distances_m,
        "is_ss": short_mask,
        "is_ls": long_mask,
    })

    channel_table["group"] = np.select(
        [channel_table["is_ss"], channel_table["is_ls"]],
        ["SS", "LS"],
        default="MID",
    )

    return channel_table

In [5]:
# %%
def build_hb_channel_table(raw_hb, subject_name, file_label):
    picks_hbo = mne.pick_types(raw_hb.info, fnirs="hbo")
    picks_hbr = mne.pick_types(raw_hb.info, fnirs="hbr")
    picks_hb = np.sort(np.concatenate([picks_hbo, picks_hbr]))

    hb_names = np.array(raw_hb.ch_names)[picks_hb]
    hb_types = np.array(raw_hb.get_channel_types())[picks_hb]
    pair_names = np.array([channel_name.split(" ")[0] for channel_name in hb_names])

    distances_all = source_detector_distances(raw_hb.info)
    distances_hb = distances_all[picks_hb]

    short_mask_all = short_channels(raw_hb.info, threshold=short_separation_threshold_m)
    short_mask = short_mask_all[picks_hb]
    long_mask = distances_hb >= long_separation_threshold_m

    hb_channel_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": hb_names,
        "pair_name": pair_names,
        "chromophore": hb_types,
        "distance_m": distances_hb,
        "is_ss": short_mask,
        "is_ls": long_mask,
    })

    hb_channel_table["group"] = np.select(
        [hb_channel_table["is_ss"], hb_channel_table["is_ls"]],
        ["SS", "LS"],
        default="MID",
    )

    return hb_channel_table

In [6]:
# %%
def apply_bad_pairs_to_hb(raw_hb, bad_pair_names):
    raw_hb = raw_hb.copy()
    hb_bad_channel_names = []

    for channel_name in raw_hb.ch_names:
        pair_name = channel_name.split(" ")[0]
        if pair_name in bad_pair_names:
            hb_bad_channel_names.append(channel_name)

    raw_hb.info["bads"] = hb_bad_channel_names
    return raw_hb

In [7]:
# %%
def make_epochs_from_raw_hb(raw_hb):
    events, event_id = mne.events_from_annotations(raw_hb, verbose=False)

    epochs_hb = mne.Epochs(
        raw_hb,
        events=events,
        event_id=event_id,
        tmin=epoch_tmin,
        tmax=epoch_tmax,
        baseline=baseline_window,
        preload=True,
        detrend=None,
        reject_by_annotation=False,
        verbose=False,
    )

    return epochs_hb, events, event_id

In [8]:
# %%
def compute_channel_midpoint(raw_hb, channel_name):
    channel_index = raw_hb.ch_names.index(channel_name)
    channel_loc = raw_hb.info["chs"][channel_index]["loc"]

    source_xyz = channel_loc[3:6]
    detector_xyz = channel_loc[6:9]
    midpoint_xyz = (source_xyz + detector_xyz) / 2.0

    return midpoint_xyz

In [9]:
# %%
def find_nearest_short_channel(raw_hb, long_channel_name):
    hb_channel_table = build_hb_channel_table(raw_hb, subject_name, file_label)

    long_channel_row = hb_channel_table.loc[
        hb_channel_table["channel_name"] == long_channel_name
    ].iloc[0]

    long_chromophore = long_channel_row["chromophore"]

    short_channel_candidates = hb_channel_table.loc[
        (hb_channel_table["group"] == "SS") &
        (hb_channel_table["chromophore"] == long_chromophore),
        "channel_name",
    ].tolist()

    if len(short_channel_candidates) == 0:
        return None

    long_midpoint = compute_channel_midpoint(raw_hb, long_channel_name)

    nearest_channel_name = None
    nearest_distance = np.inf

    for short_channel_name in short_channel_candidates:
        short_midpoint = compute_channel_midpoint(raw_hb, short_channel_name)
        midpoint_distance = np.linalg.norm(long_midpoint - short_midpoint)

        if midpoint_distance < nearest_distance:
            nearest_distance = midpoint_distance
            nearest_channel_name = short_channel_name

    return nearest_channel_name

In [10]:
# %%
def build_average_short_regressor(raw_hb, chromophore):
    hb_channel_table = build_hb_channel_table(raw_hb, subject_name, file_label)

    short_channel_names = hb_channel_table.loc[
        (hb_channel_table["group"] == "SS") &
        (hb_channel_table["chromophore"] == chromophore),
        "channel_name",
    ].tolist()

    if len(short_channel_names) == 0:
        return None

    short_channel_data = raw_hb.copy().pick(short_channel_names).get_data()
    mean_short_signal = short_channel_data.mean(axis=0)

    return mean_short_signal

In [11]:
# %%
def compute_channel_effect_table(epochs_hb, raw_hb, subject_name, file_label, pipeline_label):
    hb_channel_table = build_hb_channel_table(raw_hb, subject_name, file_label)

    ls_hbo_channel_names = hb_channel_table.loc[
        (hb_channel_table["group"] == "LS") &
        (hb_channel_table["chromophore"] == "hbo"),
        "channel_name",
    ].tolist()

    baseline_time_mask = (epochs_hb.times >= baseline_window[0]) & (epochs_hb.times <= baseline_window[1])
    response_time_mask = (epochs_hb.times >= response_window[0]) & (epochs_hb.times <= response_window[1])

    channel_effect_rows = []

    for channel_name in ls_hbo_channel_names:
        if channel_name in raw_hb.info["bads"]:
            continue

        single_channel_epochs = epochs_hb.copy().pick([channel_name])
        single_channel_data = single_channel_epochs.get_data()[:, 0, :]

        baseline_values = single_channel_data[:, baseline_time_mask].mean(axis=1)
        response_values = single_channel_data[:, response_time_mask].mean(axis=1)
        effect_values = response_values - baseline_values

        channel_effect_rows.append({
            "subject": subject_name,
            "file_label": file_label,
            "pipeline_label": pipeline_label,
            "channel_name": channel_name,
            "pair_name": channel_name.split(" ")[0],
            "mean_baseline": baseline_values.mean(),
            "mean_response": response_values.mean(),
            "mean_effect_size": effect_values.mean(),
            "std_effect_size": effect_values.std(),
        })

    channel_effect_df = pd.DataFrame(channel_effect_rows)
    channel_effect_df = channel_effect_df.sort_values("mean_effect_size", ascending=False).reset_index(drop=True)

    return channel_effect_df

In [12]:
# %%
def build_mean_epoch_plot_table(epochs_hb, selected_channel_names, pipeline_label):
    available_channel_names = [channel_name for channel_name in selected_channel_names if channel_name in epochs_hb.ch_names]

    hbo_channel_names = [channel_name for channel_name in available_channel_names if channel_name.endswith("hbo")]
    hbr_channel_names = [channel_name for channel_name in available_channel_names if channel_name.endswith("hbr")]

    plot_rows = []

    if len(hbo_channel_names) > 0:
        hbo_data = epochs_hb.copy().pick(hbo_channel_names).get_data()
        mean_hbo_time_course = hbo_data.mean(axis=(0, 1))

        plot_rows.append(pd.DataFrame({
            "time_s": epochs_hb.times,
            "signal": mean_hbo_time_course,
            "chromophore": "HbO",
            "pipeline_label": pipeline_label,
        }))

    if len(hbr_channel_names) > 0:
        hbr_data = epochs_hb.copy().pick(hbr_channel_names).get_data()
        mean_hbr_time_course = hbr_data.mean(axis=(0, 1))

        plot_rows.append(pd.DataFrame({
            "time_s": epochs_hb.times,
            "signal": mean_hbr_time_course,
            "chromophore": "HbR",
            "pipeline_label": pipeline_label,
        }))

    mean_epoch_plot_table = pd.concat(plot_rows, ignore_index=True)
    return mean_epoch_plot_table

In [13]:
# %%
def find_first_matching_column(column_names, candidate_names):
    for candidate_name in candidate_names:
        for column_name in column_names:
            if column_name == candidate_name:
                return column_name
    return None

In [14]:
# %%
def standardize_glm_dataframe(glm_df):
    glm_df = glm_df.copy()
    glm_df.columns = [str(column_name).strip().lower() for column_name in glm_df.columns]
    return glm_df

In [15]:
# %%
def get_task_regressor_names(design_matrix):
    task_regressor_names = []

    for column_name in design_matrix.columns:
        if column_name == "constant":
            continue
        if str(column_name).startswith("drift"):
            continue
        if str(column_name).startswith("ss_"):
            continue
        task_regressor_names.append(column_name)

    return task_regressor_names

In [16]:
# %%
def build_design_matrix_for_channel(raw_hb, long_channel_name, hrf_model, include_short_regressor):
    design_matrix = make_first_level_design_matrix(
        raw_hb,
        stim_dur=task_duration_s,
        drift_model="cosine",
        high_pass=drift_high_pass_hz,
        hrf_model=hrf_model,
        fir_delays=fir_delays_s if hrf_model == "fir" else None,
    )

    if include_short_regressor:
        chromophore = "hbo" if long_channel_name.endswith("hbo") else "hbr"
        nearest_short_channel_name = find_nearest_short_channel(raw_hb, long_channel_name)

        if nearest_short_channel_name is not None:
            nearest_short_signal = raw_hb.copy().pick([nearest_short_channel_name]).get_data()[0]
            design_matrix[f"ss_{chromophore}"] = nearest_short_signal
        else:
            average_short_signal = build_average_short_regressor(raw_hb, chromophore)

            if average_short_signal is not None:
                design_matrix[f"ss_{chromophore}"] = average_short_signal

    return design_matrix

In [17]:
# %%
def extract_canonical_glm_metrics(glm_df, task_regressor_name, channel_name, pipeline_label):
    glm_df = standardize_glm_dataframe(glm_df)

    condition_column_name = find_first_matching_column(
        glm_df.columns,
        ["condition", "regressor", "variable", "name"]
    )

    beta_column_name = find_first_matching_column(
        glm_df.columns,
        ["theta", "beta", "coef", "estimate", "effect"]
    )

    t_column_name = find_first_matching_column(
        glm_df.columns,
        ["t", "t_value", "tstat", "t_stat"]
    )

    p_column_name = find_first_matching_column(
        glm_df.columns,
        ["p_value", "pvalue", "p"]
    )

    if condition_column_name is None or beta_column_name is None or t_column_name is None:
        raise ValueError(f"Could not find expected GLM columns. Found columns: {glm_df.columns.tolist()}")

    canonical_rows = glm_df.loc[glm_df[condition_column_name].astype(str) == str(task_regressor_name)]

    if len(canonical_rows) == 0:
        canonical_rows = glm_df.loc[
            glm_df[condition_column_name].astype(str).str.contains(str(task_regressor_name), regex=False)
        ]

    if len(canonical_rows) == 0:
        raise ValueError(f"Could not find canonical task regressor row for {task_regressor_name}")

    canonical_row = canonical_rows.iloc[0]

    chromophore = "hbo" if channel_name.endswith("hbo") else "hbr"

    result_row = {
        "pipeline_label": pipeline_label,
        "channel_name": channel_name,
        "pair_name": channel_name.split(" ")[0],
        "chromophore": chromophore,
        "task_regressor": str(task_regressor_name),
        "beta": float(canonical_row[beta_column_name]),
        "t_value": float(canonical_row[t_column_name]),
    }

    if p_column_name is not None:
        result_row["p_value"] = float(canonical_row[p_column_name])

    return result_row

In [18]:
# %%
def extract_fir_glm_rows(glm_df, fir_regressor_names, channel_name, pipeline_label):
    glm_df = standardize_glm_dataframe(glm_df)

    condition_column_name = find_first_matching_column(
        glm_df.columns,
        ["condition", "regressor", "variable", "name"]
    )

    beta_column_name = find_first_matching_column(
        glm_df.columns,
        ["theta", "beta", "coef", "estimate", "effect"]
    )

    if condition_column_name is None or beta_column_name is None:
        raise ValueError(f"Could not find expected FIR GLM columns. Found columns: {glm_df.columns.tolist()}")

    chromophore = "hbo" if channel_name.endswith("hbo") else "hbr"

    fir_rows = []

    for fir_regressor_name in fir_regressor_names:
        matching_rows = glm_df.loc[
            glm_df[condition_column_name].astype(str) == str(fir_regressor_name)
        ]

        if len(matching_rows) == 0:
            matching_rows = glm_df.loc[
                glm_df[condition_column_name].astype(str).str.contains(str(fir_regressor_name), regex=False)
            ]

        if len(matching_rows) == 0:
            continue

        fir_row = matching_rows.iloc[0]

        delay_string = str(fir_regressor_name).split("_")[-1]
        delay_value_s = float(delay_string)

        fir_rows.append({
            "pipeline_label": pipeline_label,
            "channel_name": channel_name,
            "pair_name": channel_name.split(" ")[0],
            "chromophore": chromophore,
            "fir_regressor": str(fir_regressor_name),
            "delay_s": delay_value_s,
            "beta": float(fir_row[beta_column_name]),
        })

    fir_df = pd.DataFrame(fir_rows)
    return fir_df

In [19]:
# %%
def compute_fwhm_from_fir(delays_s, beta_values, chromophore):
    delays_s = np.array(delays_s, dtype=float)
    beta_values = np.array(beta_values, dtype=float)

    if len(beta_values) == 0:
        return np.nan

    if chromophore == "hbo":
        peak_index = int(np.argmax(beta_values))
        peak_value = beta_values[peak_index]
        if peak_value <= 0:
            return np.nan
        half_height = peak_value / 2.0
        above_half_mask = beta_values >= half_height
    else:
        peak_index = int(np.argmin(beta_values))
        peak_value = beta_values[peak_index]
        if peak_value >= 0:
            return np.nan
        half_height = peak_value / 2.0
        above_half_mask = beta_values <= half_height

    if above_half_mask.sum() < 2:
        return np.nan

    width_s = delays_s[above_half_mask].max() - delays_s[above_half_mask].min()
    return width_s

In [39]:
# %%
def preprocess_raw_to_hb(raw_cw, pipeline_label):
    raw_od = optical_density(raw_cw.copy())

    sci_values = scalp_coupling_index(raw_od)
    cw_channel_table = build_cw_channel_table(raw_cw, subject_name, file_label)

    sci_table = pd.DataFrame({
        "channel_name": np.array(raw_cw.ch_names)[get_cw_channel_indices(raw_cw)],
        "pair_name": cw_channel_table["pair_name"].values,
        "sci": sci_values,
    })

    bad_pair_names = sci_table.loc[sci_table["sci"] < sci_threshold, "pair_name"].unique().tolist()

    if pipeline_label == "PnoMC":
        processed_od = raw_od.copy()
    else:
        processed_od = tddr(raw_od.copy())

    processed_od = processed_od.copy().filter(filter_low_hz, filter_high_hz, verbose=False)
    raw_hb = beer_lambert_law(processed_od, ppf=ppf_value)
    raw_hb = apply_bad_pairs_to_hb(raw_hb, bad_pair_names)

    return {
        "pipeline_label": pipeline_label,
        "raw_od": processed_od,
        "raw_hb": raw_hb,
        "sci_table": sci_table,
        "bad_pair_names": bad_pair_names,
    }

In [40]:
# %%
def run_canonical_glm_for_channel(raw_hb, channel_name, pipeline_label, include_short_regressor):
    design_matrix = build_design_matrix_for_channel(
        raw_hb=raw_hb,
        long_channel_name=channel_name,
        hrf_model="glover",
        include_short_regressor=include_short_regressor,
    )

    task_regressor_names = get_task_regressor_names(design_matrix)
    task_regressor_name = task_regressor_names[0]

    single_channel_raw = raw_hb.copy().pick([channel_name])
    glm_result = run_glm(single_channel_raw, design_matrix, noise_model="ar1")
    glm_df = glm_result.to_dataframe()

    canonical_metrics = extract_canonical_glm_metrics(
        glm_df=glm_df,
        task_regressor_name=task_regressor_name,
        channel_name=channel_name,
        pipeline_label=pipeline_label,
    )

    return canonical_metrics

In [41]:
# %%
def run_fir_glm_for_channel(raw_hb, channel_name, pipeline_label, include_short_regressor):
    design_matrix = build_design_matrix_for_channel(
        raw_hb=raw_hb,
        long_channel_name=channel_name,
        hrf_model="fir",
        include_short_regressor=include_short_regressor,
    )

    fir_regressor_names = get_task_regressor_names(design_matrix)

    single_channel_raw = raw_hb.copy().pick([channel_name])
    glm_result = run_glm(single_channel_raw, design_matrix, noise_model="ar1")
    glm_df = glm_result.to_dataframe()

    fir_df = extract_fir_glm_rows(
        glm_df=glm_df,
        fir_regressor_names=fir_regressor_names,
        channel_name=channel_name,
        pipeline_label=pipeline_label,
    )

    return fir_df

## Load one subject and one file

In [42]:
raw_cw = mne.io.read_raw_snirf(file_path, preload=True, verbose=False)
raw_cw

/tmp/ipykernel_387316/1477888418.py:1: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_cw = mne.io.read_raw_snirf(file_path, preload=True, verbose=False)


<RawSNIRF | resting_hrf_50.snirf, 112 x 36800 (736.0 s), ~31.6 MB, data loaded>

In [43]:
# %%
raw_cw.annotations

<Annotations | 36 segments: 1 (36)>

## Build the four "realistc" pipelines

In [44]:
# %%
pipeline_results = []

for pipeline_label in pipeline_labels:
    pipeline_result = preprocess_raw_to_hb(raw_cw, pipeline_label)
    pipeline_results.append(pipeline_result)

len(pipeline_results)

4

In [45]:
# %%
sci_summary_rows = []

for pipeline_result in pipeline_results:
    sci_table = pipeline_result["sci_table"]

    sci_summary_rows.append({
        "pipeline_label": pipeline_result["pipeline_label"],
        "n_cw_channels": len(sci_table),
        "mean_sci": sci_table["sci"].mean(),
        "median_sci": sci_table["sci"].median(),
        "min_sci": sci_table["sci"].min(),
        "max_sci": sci_table["sci"].max(),
        "n_pairs_below_threshold": len(pipeline_result["bad_pair_names"]),
    })

sci_summary_df = pd.DataFrame(sci_summary_rows)
sci_summary_df

,pipeline_label,n_cw_channels,mean_sci,median_sci,min_sci,max_sci,n_pairs_below_threshold
0,Pref,112,0.797839,0.860094,0.089575,0.978122,3
1,PnoSS,112,0.797839,0.860094,0.089575,0.978122,3
2,PnoMC,112,0.797839,0.860094,0.089575,0.978122,3
3,PFIR,112,0.797839,0.860094,0.089575,0.978122,3


## Select target ROI pairs from Pref
### Until the injected-channel metadata is wired in directly, use the top-M fallback.

In [46]:
# %%
reference_pipeline_result = [result for result in pipeline_results if result["pipeline_label"] == "Pref"][0]
reference_raw_hb = reference_pipeline_result["raw_hb"]

reference_epochs_hb, reference_events, reference_event_id = make_epochs_from_raw_hb(reference_raw_hb)

reference_channel_effect_df = compute_channel_effect_table(
    epochs_hb=reference_epochs_hb,
    raw_hb=reference_raw_hb,
    subject_name=subject_name,
    file_label=file_label,
    pipeline_label="Pref",
)

reference_channel_effect_df.head(15)

,subject,file_label,pipeline_label,channel_name,pair_name,mean_baseline,mean_response,mean_effect_size,std_effect_size
0,Subj100,hrf_50,Pref,S10_D18 hbo,S10_D18,1.063588e-21,0.000014,0.000014,0.000018
1,Subj100,hrf_50,Pref,S5_D4 hbo,S5_D4,-3.956516e-22,0.000013,0.000013,0.000025
2,Subj100,hrf_50,Pref,S11_D25 hbo,S11_D25,-2.100468e-22,0.000012,0.000012,0.000013
3,Subj100,hrf_50,Pref,S3_D9 hbo,S3_D9,3.154318e-22,0.000012,0.000012,0.000023
4,Subj100,hrf_50,Pref,S10_D24 hbo,S10_D24,-7.377430e-22,0.000012,0.000012,0.000015
5,Subj100,hrf_50,Pref,S4_D3 hbo,S4_D3,1.017404e-21,0.000011,0.000011,0.000034
6,Subj100,hrf_50,Pref,S2_D7 hbo,S2_D7,-6.367451e-22,0.000011,0.000011,0.000018
7,Subj100,hrf_50,Pref,S3_D3 hbo,S3_D3,7.385626e-23,0.000011,0.000011,0.000030
8,Subj100,hrf_50,Pref,S2_D2 hbo,S2_D2,-1.496409e-22,0.000011,0.000011,0.000021
9,Subj100,hrf_50,Pref,S12_D19 hbo,S12_D19,2.890373e-22,0.000011,0.000011,0.000024


In [47]:
# %%
reference_channel_rank_figure = px.bar(
    reference_channel_effect_df.head(15),
    x="channel_name",
    y="mean_effect_size",
    title=f"{subject_name} {file_label}: top LS HbO channels from Pref",
)

reference_channel_rank_figure.update_layout(
    width=1100,
    height=500,
    xaxis_tickangle=-45,
)

reference_channel_rank_figure.show()

In [48]:
# %%
top_hbo_channel_names = reference_channel_effect_df.head(top_m_pairs)["channel_name"].tolist()
target_pair_names = [channel_name.replace(" hbo", "") for channel_name in top_hbo_channel_names]
top_hbr_channel_names = [pair_name + " hbr" for pair_name in target_pair_names]

selected_channel_names = top_hbo_channel_names + top_hbr_channel_names

print("Target pair names:")
print(target_pair_names)

print("\nSelected channel names:")
print(selected_channel_names)

Target pair names:
['S10_D18', 'S5_D4', 'S11_D25', 'S3_D9', 'S10_D24']

Selected channel names:
['S10_D18 hbo', 'S5_D4 hbo', 'S11_D25 hbo', 'S3_D9 hbo', 'S10_D24 hbo', 'S10_D18 hbr', 'S5_D4 hbr', 'S11_D25 hbr', 'S3_D9 hbr', 'S10_D24 hbr']


## Compare processed Hb responses across the four pipelines

In [49]:
# %%
epoch_results = []

for pipeline_result in pipeline_results:
    pipeline_label = pipeline_result["pipeline_label"]
    raw_hb = pipeline_result["raw_hb"]

    epochs_hb, events, event_id = make_epochs_from_raw_hb(raw_hb)

    epoch_results.append({
        "pipeline_label": pipeline_label,
        "raw_hb": raw_hb,
        "epochs_hb": epochs_hb,
        "events": events,
        "event_id": event_id,
    })

len(epoch_results)

4

In [50]:
# %%
pipeline_plot_tables = []

for epoch_result in epoch_results:
    mean_epoch_plot_table = build_mean_epoch_plot_table(
        epochs_hb=epoch_result["epochs_hb"],
        selected_channel_names=selected_channel_names,
        pipeline_label=epoch_result["pipeline_label"],
    )

    pipeline_plot_tables.append(mean_epoch_plot_table)

pipeline_plot_df = pd.concat(pipeline_plot_tables, ignore_index=True)
pipeline_plot_df.head()

,time_s,signal,chromophore,pipeline_label
0,-5.00,0.000001,HbO,Pref
1,-4.98,0.000001,HbO,Pref
2,-4.96,0.000001,HbO,Pref
3,-4.94,0.000001,HbO,Pref
4,-4.92,0.000001,HbO,Pref


In [51]:
# %%
pipeline_epoch_figure = px.line(
    pipeline_plot_df,
    x="time_s",
    y="signal",
    color="pipeline_label",
    facet_row="chromophore",
    title=f"{subject_name} {file_label}: event-locked Hb responses across true pipelines",
)

pipeline_epoch_figure.update_layout(
    width=1100,
    height=700,
)

pipeline_epoch_figure.add_vline(x=0.0)
pipeline_epoch_figure.show()
pipeline_epoch_figure.write_html(
    output_figures_dir / f"{subject_name.lower()}_{file_label}_true_pipeline_epoch_comparison.html"
)

In [52]:
# %%
pipeline_epoch_metric_rows = []

for epoch_result in epoch_results:
    pipeline_label = epoch_result["pipeline_label"]
    epochs_hb = epoch_result["epochs_hb"]

    available_hbo_channel_names = [channel_name for channel_name in top_hbo_channel_names if channel_name in epochs_hb.ch_names]
    available_hbr_channel_names = [channel_name for channel_name in top_hbr_channel_names if channel_name in epochs_hb.ch_names]

    hbo_data = epochs_hb.copy().pick(available_hbo_channel_names).get_data()
    hbr_data = epochs_hb.copy().pick(available_hbr_channel_names).get_data()

    baseline_time_mask = (epochs_hb.times >= baseline_window[0]) & (epochs_hb.times <= baseline_window[1])
    response_time_mask = (epochs_hb.times >= response_window[0]) & (epochs_hb.times <= response_window[1])

    hbo_baseline = hbo_data[:, :, baseline_time_mask].mean()
    hbo_response = hbo_data[:, :, response_time_mask].mean()
    hbr_baseline = hbr_data[:, :, baseline_time_mask].mean()
    hbr_response = hbr_data[:, :, response_time_mask].mean()

    pipeline_epoch_metric_rows.append({
        "subject": subject_name,
        "file_label": file_label,
        "pipeline_label": pipeline_label,
        "hbo_mean_effect": hbo_response - hbo_baseline,
        "hbr_mean_effect": hbr_response - hbr_baseline,
        "n_events": len(epochs_hb),
        "n_hbo_channels": len(available_hbo_channel_names),
        "n_hbr_channels": len(available_hbr_channel_names),
    })

pipeline_epoch_metric_df = pd.DataFrame(pipeline_epoch_metric_rows)
pipeline_epoch_metric_df

,subject,file_label,pipeline_label,hbo_mean_effect,hbr_mean_effect,n_events,n_hbo_channels,n_hbr_channels
0,Subj100,hrf_50,Pref,0.000013,-0.000005,35,5,5
1,Subj100,hrf_50,PnoSS,0.000013,-0.000005,35,5,5
2,Subj100,hrf_50,PnoMC,0.000015,-0.000003,35,5,5
3,Subj100,hrf_50,PFIR,0.000013,-0.000005,35,5,5


## Canonical GLM for all four pipelines
- Pref, PnoSS, and PnoMC differ in whether short-channel regressors and TDDR are used.
-  PFIR keeps the same preprocessing as Pref, and also gets a canonical GLM so its amplitudes are comparable.

In [53]:
# %%
canonical_channel_rows = []
fir_channel_rows = []

for pipeline_result in pipeline_results:
    pipeline_label = pipeline_result["pipeline_label"]
    raw_hb = pipeline_result["raw_hb"]

    for channel_name in selected_channel_names:
        if channel_name not in raw_hb.ch_names:
            continue

        if channel_name in raw_hb.info["bads"]:
            continue

        include_short_regressor = pipeline_label != "PnoSS"

        canonical_metrics = run_canonical_glm_for_channel(
            raw_hb=raw_hb,
            channel_name=channel_name,
            pipeline_label=pipeline_label,
            include_short_regressor=include_short_regressor,
        )

        canonical_channel_rows.append(canonical_metrics)

        if pipeline_label == "PFIR":
            fir_channel_df = run_fir_glm_for_channel(
                raw_hb=raw_hb,
                channel_name=channel_name,
                pipeline_label=pipeline_label,
                include_short_regressor=True,
            )

            if len(fir_channel_df) > 0:
                fir_channel_rows.append(fir_channel_df)

canonical_channel_df = pd.DataFrame(canonical_channel_rows)

if len(fir_channel_rows) > 0:
    fir_channel_df = pd.concat(fir_channel_rows, ignore_index=True)
else:
    fir_channel_df = pd.DataFrame()

canonical_channel_df.head(12)

,pipeline_label,channel_name,pair_name,chromophore,task_regressor,beta,t_value,p_value
0,Pref,S10_D18 hbo,S10_D18,hbo,1,0.000037,54.448704,1.611994e-20
1,Pref,S5_D4 hbo,S5_D4,hbo,1,0.000030,30.336771,3.031488e-16
2,Pref,S11_D25 hbo,S11_D25,hbo,1,0.000037,79.043171,2.923826e-23
3,Pref,S3_D9 hbo,S3_D9,hbo,1,0.000032,45.539865,3.295625e-19
4,Pref,S10_D24 hbo,S10_D24,hbo,1,0.000032,58.368049,4.974651e-21
5,Pref,S10_D18 hbr,S10_D18,hbr,1,-0.000009,-40.058524,2.860743e-18
6,Pref,S5_D4 hbr,S5_D4,hbr,1,-0.000014,-22.315406,4.959768e-14
7,Pref,S11_D25 hbr,S11_D25,hbr,1,-0.000011,-48.997826,9.582208e-20
8,Pref,S3_D9 hbr,S3_D9,hbr,1,-0.000011,-35.260140,2.442434e-17
9,Pref,S10_D24 hbr,S10_D24,hbr,1,-0.000011,-42.835444,9.252090e-19


In [54]:
# %%
canonical_roi_summary_df = (
    canonical_channel_df
    .groupby(["pipeline_label", "chromophore"], as_index=False)
    .agg(
        roi_mean_beta=("beta", "mean"),
        roi_std_beta=("beta", "std"),
        roi_mean_t=("t_value", "mean"),
        roi_max_t=("t_value", "max"),
        n_channels=("channel_name", "count"),
    )
)

canonical_roi_summary_df

,pipeline_label,chromophore,roi_mean_beta,roi_std_beta,roi_mean_t,roi_max_t,n_channels
0,PFIR,hbo,0.000034,0.000003,53.547312,79.043171,5
1,PFIR,hbr,-0.000011,0.000002,-37.893468,-22.315406,5
2,PnoMC,hbo,0.000034,0.000039,38.895436,62.596093,5
3,PnoMC,hbr,-0.000013,0.000005,-19.467128,-13.069092,5
4,PnoSS,hbo,0.000032,0.000005,40.887745,52.131235,5
5,PnoSS,hbr,-0.000011,0.000003,-32.859672,-23.024005,5
6,Pref,hbo,0.000034,0.000003,53.547312,79.043171,5
7,Pref,hbr,-0.000011,0.000002,-37.893468,-22.315406,5


In [55]:
# %%
canonical_beta_figure = px.bar(
    canonical_roi_summary_df,
    x="pipeline_label",
    y="roi_mean_beta",
    color="chromophore",
    barmode="group",
    title=f"{subject_name} {file_label}: canonical GLM ROI mean beta across pipelines",
)

canonical_beta_figure.update_layout(
    width=1000,
    height=500,
)

canonical_beta_figure.show()
canonical_beta_figure.write_html(
    output_figures_dir / f"{subject_name.lower()}_{file_label}_canonical_roi_beta_comparison.html"
)

In [56]:
# %%
canonical_t_figure = px.bar(
    canonical_roi_summary_df,
    x="pipeline_label",
    y="roi_mean_t",
    color="chromophore",
    barmode="group",
    title=f"{subject_name} {file_label}: canonical GLM ROI mean t across pipelines",
)

canonical_t_figure.update_layout(
    width=1000,
    height=500,
)

canonical_t_figure.show()
canonical_t_figure.write_html(
    output_figures_dir / f"{subject_name.lower()}_{file_label}_canonical_roi_t_comparison.html"
)

## FIR shape metrics for PFIR

In [57]:
# %%
fir_channel_df.head(12)

,pipeline_label,channel_name,pair_name,chromophore,fir_regressor,delay_s,beta
0,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_0,0.0,-2.501851e-06
1,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_1,1.0,-2.646746e-09
2,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_2,2.0,-9.127592e-10
3,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_3,3.0,1.528619e-11
4,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_4,4.0,1.708627e-11
5,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_5,5.0,-1.030570e-09
6,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_6,6.0,-3.151437e-09
7,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_7,7.0,-6.314793e-09
8,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_8,8.0,-1.045885e-08
9,PFIR,S10_D18 hbo,S10_D18,hbo,1_delay_9,9.0,-1.553963e-08


In [58]:
# %%
fir_roi_timecourse_df = (
    fir_channel_df
    .groupby(["pipeline_label", "chromophore", "delay_s"], as_index=False)
    .agg(
        roi_mean_beta=("beta", "mean"),
        roi_std_beta=("beta", "std"),
    )
)

fir_roi_timecourse_df.head(12)

,pipeline_label,chromophore,delay_s,roi_mean_beta,roi_std_beta
0,PFIR,hbo,0.0,-3.656209e-06,9.378532e-07
1,PFIR,hbo,1.0,-5.416119e-09,2.108179e-08
2,PFIR,hbo,2.0,-6.739030e-10,1.967646e-08
3,PFIR,hbo,3.0,3.644177e-09,1.959549e-08
4,PFIR,hbo,4.0,7.435252e-09,2.089832e-08
5,PFIR,hbo,5.0,1.059157e-08,2.332482e-08
6,PFIR,hbo,6.0,1.305387e-08,2.647639e-08
7,PFIR,hbo,7.0,1.476319e-08,2.995876e-08
8,PFIR,hbo,8.0,1.569875e-08,3.345922e-08
9,PFIR,hbo,9.0,1.586305e-08,3.678095e-08


In [59]:
# %%
fir_roi_figure = px.line(
    fir_roi_timecourse_df,
    x="delay_s",
    y="roi_mean_beta",
    color="chromophore",
    title=f"{subject_name} {file_label}: PFIR ROI FIR beta time course",
)

fir_roi_figure.update_layout(
    width=1000,
    height=500,
)

fir_roi_figure.show()
fir_roi_figure.write_html(
    output_figures_dir / f"{subject_name.lower()}_{file_label}_pfir_roi_timecourse.html"
)

In [60]:
# %%
fir_shape_rows = []

for chromophore in ["hbo", "hbr"]:
    chromophore_df = fir_roi_timecourse_df.loc[
        fir_roi_timecourse_df["chromophore"] == chromophore
    ].sort_values("delay_s")

    delays_s = chromophore_df["delay_s"].to_numpy()
    beta_values = chromophore_df["roi_mean_beta"].to_numpy()

    if chromophore == "hbo":
        peak_index = int(np.argmax(beta_values))
    else:
        peak_index = int(np.argmin(beta_values))

    peak_delay_s = float(delays_s[peak_index])
    peak_beta = float(beta_values[peak_index])
    fwhm_s = compute_fwhm_from_fir(delays_s, beta_values, chromophore)

    fir_shape_rows.append({
        "subject": subject_name,
        "file_label": file_label,
        "pipeline_label": "PFIR",
        "chromophore": chromophore,
        "peak_delay_s": peak_delay_s,
        "peak_beta": peak_beta,
        "fwhm_s": fwhm_s,
    })

fir_shape_summary_df = pd.DataFrame(fir_shape_rows)
fir_shape_summary_df

,subject,file_label,pipeline_label,chromophore,peak_delay_s,peak_beta,fwhm_s
0,Subj100,hrf_50,PFIR,hbo,25.0,7.743257e-07,NaN
1,Subj100,hrf_50,PFIR,hbr,18.0,3.178927e-09,NaN


## Final summary tables

In [61]:
# %%
pipeline_final_summary_df = canonical_roi_summary_df.copy()

pipeline_final_summary_df = pipeline_final_summary_df.merge(
    pipeline_epoch_metric_df[["pipeline_label", "hbo_mean_effect", "hbr_mean_effect", "n_events"]],
    on="pipeline_label",
    how="left",
)

pipeline_final_summary_df

,pipeline_label,chromophore,roi_mean_beta,roi_std_beta,roi_mean_t,roi_max_t,n_channels,hbo_mean_effect,hbr_mean_effect,n_events
0,PFIR,hbo,0.000034,0.000003,53.547312,79.043171,5,0.000013,-0.000005,35
1,PFIR,hbr,-0.000011,0.000002,-37.893468,-22.315406,5,0.000013,-0.000005,35
2,PnoMC,hbo,0.000034,0.000039,38.895436,62.596093,5,0.000015,-0.000003,35
3,PnoMC,hbr,-0.000013,0.000005,-19.467128,-13.069092,5,0.000015,-0.000003,35
4,PnoSS,hbo,0.000032,0.000005,40.887745,52.131235,5,0.000013,-0.000005,35
5,PnoSS,hbr,-0.000011,0.000003,-32.859672,-23.024005,5,0.000013,-0.000005,35
6,Pref,hbo,0.000034,0.000003,53.547312,79.043171,5,0.000013,-0.000005,35
7,Pref,hbr,-0.000011,0.000002,-37.893468,-22.315406,5,0.000013,-0.000005,35


## Save outputs

In [62]:
"""
# %%
reference_channel_effect_df.to_csv(
    output_tables_dir / f"{subject_name.lower()}_{file_label}_reference_channel_effects_true_pipeline.csv",
    index=False,
)

pipeline_epoch_metric_df.to_csv(
    output_tables_dir / f"{subject_name.lower()}_{file_label}_pipeline_epoch_metrics_true_pipeline.csv",
    index=False,
)

canonical_channel_df.to_csv(
    output_tables_dir / f"{subject_name.lower()}_{file_label}_canonical_channel_metrics_true_pipeline.csv",
    index=False,
)

canonical_roi_summary_df.to_csv(
    output_tables_dir / f"{subject_name.lower()}_{file_label}_canonical_roi_summary_true_pipeline.csv",
    index=False,
)

if len(fir_channel_df) > 0:
    fir_channel_df.to_csv(
        output_tables_dir / f"{subject_name.lower()}_{file_label}_pfir_channel_timecourse.csv",
        index=False,
    )

    fir_shape_summary_df.to_csv(
        output_tables_dir / f"{subject_name.lower()}_{file_label}_pfir_shape_summary.csv",
        index=False,
    )

pipeline_final_summary_df.to_csv(
    output_tables_dir / f"{subject_name.lower()}_{file_label}_pipeline_final_summary_true_pipeline.csv",
    index=False,
)

print("Saved true pipeline notebook outputs.")
"""

'\n# %%\nreference_channel_effect_df.to_csv(\n    output_tables_dir / f"{subject_name.lower()}_{file_label}_reference_channel_effects_true_pipeline.csv",\n    index=False,\n)\n\npipeline_epoch_metric_df.to_csv(\n    output_tables_dir / f"{subject_name.lower()}_{file_label}_pipeline_epoch_metrics_true_pipeline.csv",\n    index=False,\n)\n\ncanonical_channel_df.to_csv(\n    output_tables_dir / f"{subject_name.lower()}_{file_label}_canonical_channel_metrics_true_pipeline.csv",\n    index=False,\n)\n\ncanonical_roi_summary_df.to_csv(\n    output_tables_dir / f"{subject_name.lower()}_{file_label}_canonical_roi_summary_true_pipeline.csv",\n    index=False,\n)\n\nif len(fir_channel_df) > 0:\n    fir_channel_df.to_csv(\n        output_tables_dir / f"{subject_name.lower()}_{file_label}_pfir_channel_timecourse.csv",\n        index=False,\n    )\n\n    fir_shape_summary_df.to_csv(\n        output_tables_dir / f"{subject_name.lower()}_{file_label}_pfir_shape_summary.csv",\n        index=False,\n  